In [23]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
import plotly.graph_objects as go
from google.colab import files


#  Load and preprocess data

df = pd.read_csv("HistoricalData_1746263471295.csv")

# Clean and rename columns
df.rename(columns={
    'Close/Last': 'Close',
    'Open': 'Open',
    'High': 'High',
    'Low': 'Low',
    'Volume': 'Volume'
}, inplace=True)

# Remove $ and , then convert to float
for col in ['Open', 'High', 'Low', 'Close']:
    df[col] = df[col].replace('[\$,]', '', regex=True).astype(float)
df['Volume'] = df['Volume'].replace(',', '', regex=True).astype(float)

# Process dates
df['Date'] = pd.to_datetime(df['Date'])
df.sort_values('Date', inplace=True)
df.set_index('Date', inplace=True)

#  Scale data and create sequences
features = ['Open', 'High', 'Low', 'Close', 'Volume']
scaler = MinMaxScaler()
scaled_df = scaler.fit_transform(df[features])

def create_dataset(data, time_step=60):
    X, y = [], []
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i])
        y.append(data[i, 3])  # Close is at index 3
    return np.array(X), np.array(y)

time_step = 60
X, y = create_dataset(scaled_df, time_step)

# Train-test split
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Reshape for LSTM [samples, time steps, features]
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], X_train.shape[2])
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], X_test.shape[2])

#  Build and train LSTM model
model = Sequential([
    LSTM(100, return_sequences=True, input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(0.2),
    LSTM(100, return_sequences=False),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer='adam', loss='mean_squared_error')
model.fit(X_train, y_train, epochs=50, batch_size=64, verbose=1)

# Make predictions ===
y_pred = model.predict(X_test)

# Inverse transform close prices only
scaler_close = MinMaxScaler()
scaler_close.fit(df[['Close']])
y_pred_inv = scaler_close.inverse_transform(y_pred)
y_test_inv = scaler_close.inverse_transform(y_test.reshape(-1, 1))

#  Calculate error ===
mse = mean_squared_error(y_test_inv, y_pred_inv)
mae = mean_absolute_error(y_test_inv, y_pred_inv)
print(f"\n📉 MSE: {mse:.2f}")
print(f"📈 MAE: {mae:.2f}")

#  Plot results ===
plot_dates = df.index[train_size + time_step:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=plot_dates, y=y_test_inv.flatten(), mode='lines', name='Actual Price'))
fig.add_trace(go.Scatter(x=plot_dates, y=y_pred_inv.flatten(), mode='lines', name='Predicted Price'))
fig.update_layout(title='AAPL Stock Price Prediction Using LSTM',
                  xaxis_title='Date', yaxis_title='Stock Price (USD)')
fig.show()


Epoch 1/50


/usr/local/lib/python3.11/dist-packages/keras/src/layers/rnn/rnn.py:200: UserWarning:

Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.



31/31 ━━━━━━━━━━━━━━━━━━━━ 9s 176ms/step - loss: 0.0246
Epoch 2/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 170ms/step - loss: 0.0011
Epoch 3/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - loss: 9.6511e-04
Epoch 4/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 12s 213ms/step - loss: 7.8841e-04
Epoch 5/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 218ms/step - loss: 6.9366e-04
Epoch 6/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 9s 168ms/step - loss: 7.6720e-04
Epoch 7/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - loss: 8.3279e-04
Epoch 8/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 171ms/step - loss: 7.2251e-04
Epoch 9/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 167ms/step - loss: 7.6451e-04
Epoch 10/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 11s 200ms/step - loss: 6.5717e-04
Epoch 11/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 11s 217ms/step - loss: 6.4088e-04
Epoch 12/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 9s 175ms/step - loss: 8.0918e-04
Epoch 13/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 171ms/step - loss: 6.8998e-04
Epoch 14/50
31/31 ━━━━━━━━━━━━━━━━━━━━ 10s 169ms/step - loss: 6.5753e-04
Epoch 15/